# 1. Connecting, extracting and validating

**Goal:** load the analytical views from PostgreSQL and prove the data is trustworthy before analysing it.

**Why this order?** Real analysts never start with charts. They first check that the numbers reconcile.

*What to learn:* SQLAlchemy connection from environment variables (`.env`), reading a view with `pandas.read_sql`, and checking one number through two independent routes.

In [1]:
import sys
sys.path.insert(0, "..")            # make the modules in python/ importable from this folder
import pandas as pd
import matplotlib.pyplot as plt
import visualization as viz
from data_extraction import get_view
pd.set_option("display.width", 160, "display.max_columns", 20, "display.float_format", "{:,.1f}".format)
%matplotlib inline

## Connection
Credentials come from environment variables (see `.env.example`); nothing is hard-coded.

In [2]:
from database_connection import run_query
run_query('SELECT version() AS postgres_version')

,postgres_version
0,PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on ...


## What views are available?

In [3]:
from data_extraction import VIEWS
pd.DataFrame({'view': list(VIEWS), 'contents': list(VIEWS.values())})

,view,contents
0,vw_monthly_sales,"one row per month: orders, revenue, profit, re..."
1,vw_product_performance,"one row per product: sales, margin, returns, s..."
2,vw_customer_summary,"one row per customer: value, RFM scores and se..."
3,vw_inventory_health,one row per store + product: stock level and h...
4,vw_store_performance,"one row per store: revenue, AOV, repeat rate, ..."
5,vw_category_performance,"one row per category and year: revenue, margin..."
6,vw_supplier_performance,"one row per supplier: spend, on-time %, fill rate"
7,vw_return_analysis,"one row per returned line: reason, store, cate..."
8,vw_orders,one row per order (all statuses)
9,vw_sales_lines,one row per sold order line (completed orders)


## Load the core views and look at their shape

In [4]:
from data_extraction import extract_core
data = extract_core()
pd.DataFrame({v: {'rows': len(df), 'columns': df.shape[1]} for v, df in data.items()}).T

,rows,columns
vw_monthly_sales,36,17
vw_product_performance,636,30
vw_customer_summary,9120,23
vw_inventory_health,5690,23
vw_store_performance,12,23
vw_category_performance,51,12
vw_supplier_performance,55,19
vw_return_analysis,2901,21


## Validation checks
`data_validation.run_checks` reconciles revenue, profit, units, orders and inventory value **across views and against the base tables**.

In [5]:
import data_validation
checks = data_validation.run_checks(data)
report = data_validation.report(checks)
print(f"{(report.status == 'PASS').sum()} of {len(report)} checks passed")
report

30 of 30 checks passed


,status,check,detail
0,PASS,vw_monthly_sales: non-empty with expected columns,36 rows; missing=[]
1,PASS,vw_product_performance: non-empty with expecte...,636 rows; missing=[]
2,PASS,vw_customer_summary: non-empty with expected c...,"9,120 rows; missing=[]"
3,PASS,vw_inventory_health: non-empty with expected c...,"5,690 rows; missing=[]"
4,PASS,vw_store_performance: non-empty with expected ...,12 rows; missing=[]
5,PASS,vw_category_performance: non-empty with expect...,51 rows; missing=[]
6,PASS,vw_supplier_performance: non-empty with expect...,55 rows; missing=[]
7,PASS,vw_return_analysis: non-empty with expected co...,"2,901 rows; missing=[]"
8,PASS,customer_id is unique in vw_customer_summary,
9,PASS,product_id is unique in vw_product_performance,


## Null and dtype profile of the customer view
Missing values here are expected: customers who never bought have no order dates or RFM scores.

In [6]:
c = data['vw_customer_summary']
pd.DataFrame({'dtype': c.dtypes.astype(str), 'nulls': c.isna().sum(), 'null_%': (100 * c.isna().mean()).round(1)})

,dtype,nulls,null_%
customer_id,int64,0,0.0
customer_name,str,0,0.0
email,str,305,3.3
city,str,96,1.1
state,str,96,1.1
segment,str,0,0.0
registration_date,object,0,0.0
is_active,bool,0,0.0
orders,int64,0,0.0
net_revenue,float64,0,0.0


## Headline KPIs
Defined once in `kpis.py` and tested against the SQL query `Q401`.

In [7]:
from kpis import headline_kpis
orders = get_view('vw_orders')
kpis = headline_kpis(data['vw_monthly_sales'], data['vw_customer_summary'], data['vw_inventory_health'], orders)
pd.Series(kpis, name='value').to_frame()

,value
net_revenue,"152,527,101.0"
gross_profit,"35,648,994.0"
gross_margin_pct,23.4
completed_orders,"54,260.0"
cancelled_orders,"2,315.0"
buying_customers,"7,074.0"
avg_order_value,"3,202.0"
repeat_customer_rate_pct,83.4
order_return_rate_pct,5.2
inventory_value_at_cost,"62,748,206.0"
